# 99 — Research Notes and Paper Framing

## Research Question
How should the experiments be framed as scientific claims for a paper?

## Hypothesis
The strongest paper narrative centers on behavioral evidence, early observability and cognitive threat measurement.

## Input Data
- Notes from current notebook
- Experimental checklist

## Prediction/Analysis Target
- Claims, contributions, terminology and reviewer-facing framing

## Validation Protocol
Non-executable research planning notebook.

## Expected Paper Artifact
- Paper outline
- Contribution list
- Reviewer-objection checklist


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_score
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    make_scorer,
)

from evaluation import (
    load_jsonl, 
    evaluate_binary_prediction, 
    evaluate_multiclass_attack_type,
    fit_rf_feature_importance,
    evaluate_multiclass_feature_group_ablation,
    add_akc_phase_from_attack_type,
    normalize_seed_column,
    infer_telemetry_features,
    prepare_akc_phase_df,
    run_akc_phase_leave_one_seed,
    summarize_multiclass_results,
    collect_akc_phase_predictions_leave_one_seed,
    plot_confusion_matrix_from_predictions,
    infer_early_akc_features,
    classification_report,
    run_temporal_akc_phase_classification,
    plot_temporal_akc_curves,
    adversarial_fragmentation_index,
    risk_conditioned_fragmentation,
    semantic_fragmentation_proxy,
    add_benign_normalized_fragmentation,
    evaluate_leave_one_seed_out,
    TELEMETRY_NUMERIC,
    evaluate_feature_group_ablation,
    FEATURE_GROUPS,
    evaluate_leave_one_attack_out,
    compute_permutation_importance,
    run_operational_only_leave_one_seed,
    summarize_results,
    plot_exp1_operational_comparison,
    load_tamas_jsonl,
    build_early_df_from_agent_outputs,
    run_early_detection_leave_one_seed,
    plot_early_detection_curve,
    EARLY_FEATURES_CANDIDATES,
    validate_early_df_for_attack_analysis,
    run_early_detection_by_attack_type,
    summarize_early_detection_by_attack,
    plot_metric_curves_by_attack,
    compute_temporal_gain,
    infer_numeric_features,
    keep_existing_numeric,
    run_early_detection_feature_ablation,
    plot_ablation_curves,
    telemetry_guardrail_decision,
    run_akc_phase_leave_one_attack_type_out,
)
from datasets.tamas.tamas_features import build_all_feature_tables

RESULTS_DIR = Path("results/tamas")
RAW_DIR = RESULTS_DIR / "raw"
FEATURES_DIR = RESULTS_DIR / "features"
PLOTS_DIR = RESULTS_DIR / "plots"

scenario = "education"
architecture = "centralized_tamas"
CONDITIONS = [
    "benign",
    "byzantine",
    "colluding",
    "contradicting",
    "DPI",
    "impersonation",
    "IPI",
]
SEEDS = [1, 2, 3]
MODEL_NAMES = [
    "ticlazau/meta-llama-3.1-8b-instruct:latest",
    # "qwen3",
]

9. O que eu priorizaria agora

A ordem que eu seguiria:

1. Cross-seed holdout para is_attack.
2. Ablation por grupos com leave-one-seed-out.
3. Early detection com 25/50/75/100% do trace.
4. Refinar fragmentação com risco e normalização contra benignos.
5. Criar benignos pareados automaticamente.
6. Substituir attack_success_proxy por outcomes comportamentais mais densos.
7. Adicionar uma segunda arquitetura, pelo menos single_agent_tamas, para mostrar que a telemetria não depende só de centralized_tamas.

Propagation Depth: número de agentes cujas mensagens incorporam conteúdo suspeito após o primeiro agente comprometido

Contagion Ratio: agentes afetados / total de agentes

Suspicious Influence: similaridade entre resposta final e mensagens do agente comprometido

Tool Propagation: ferramentas chamadas depois de uma mensagem suspeita

[ ] Benignos pareados por tarefa-base

[ ] 5 seeds ou pelo menos 3 com leave-one-seed-out

[ ] Attack-type classification incluindo benign

[ ] AKC/OODA phase prediction

[ ] AFI com ablação formal

[ ] Early detection por fração do trace

[ ] Leave-one-attack-out generalization

[ ] Compromised-agent localization

[ ] Telemetry-triggered intervention

[ ] Pelo menos 2 domínios - finance ou healthcare

[ ] Pelo menos 2 arquiteturas - decentralized_tamas/hybrid_tamas

Mostrar que a telemetria captura agentes comprometidos

Fazer leave-one-attack-out

Fazer validação leave-one-seed-out

Fazer ablação forte da métrica AFI

AFI full

AFI sem semantic drift

AFI sem risk signal

AFI sem functional normalization

AFI só tool features

AFI só linguistic markers

AFI só cost

1. Sair de “logs/features” e ir para “cognitive threat measurement”

A narrativa AAAI deve ser:

LLM-based multi-agent systems are cognitive-action systems. Attacks do not only change final answers; they alter how agents observe, orient, decide, coordinate, and act. We show that runtime telemetry can measure these cognitive threat phases.

2. Transformar o paper em hipótese científica

Estruture o paper em três hipóteses:

H1 — Behavioral separability

Different agentic attack classes induce distinguishable telemetry signatures.

Experimento: multiclass attack-type classification + feature ablation.

H2 — Cognitive phase observability

Telemetry features map to Agentic Kill Chain/OODA phases.

Experimento: akc_phase classification + phase-specific telemetry signatures.

H3 — Early threat detection

Partial traces reveal attack phase before final output or tool execution.

Experimento: early detection por trace_fraction.

3. Adicionar intervenção simples

AAAI valoriza contribuição que não apenas analisa, mas também age.

Adicione um experimento de intervenção: Telemetry-triggered guardrail

Exemplos:

* se injection_marker_count > 0, acionar verifier;

* se contradiction_marker_count alto, bloquear consenso;

* se avg_pairwise_message_similarity muito alta em cenário colluding, exigir verificação independente;

* se tool_call_entropy ou tool suspeita aparecer, bloquear tool execution.

In [ ]:
episode_df_lst["guardrail_action"] = episode_df_lst.apply(
    telemetry_guardrail_decision,
    axis=1,
)

display(
    episode_df_lst.groupby(["attack_type", "guardrail_action"])
    .size()
    .reset_index(name="n")
)

Telemetry as Behavioral Evidence: Measuring Cognitive Threat Phases in LLM-based Multi-Agent Systems

Claim central

Runtime telemetry provides observable evidence of cognitive threat phases in LLM-based multi-agent workflows.

Contribuições

1. Framework conceitual: mapeia telemetria de execução para fases da Agentic Kill Chain/OODA.

2. Behavioral Telemetry Profiles: métricas de custo, coordenação, tool-use, contradição, fragmentação semântica e propagação.

3. Evidência empírica: mostra que ataques multiagentes produzem assinaturas de telemetria distinguíveis.

4. Detecção precoce: avalia se traces parciais predizem classe/fase do ataque.

5. Intervenção: propõe guardrails acionados por telemetria.

Byzantine Agent - An agent may provide factually incorrect answers, intentionally sabotage tool usage, or inject irrelevant noise into the communication.